# 27 — FIM Sync Stability: Hysteresis, Lyapunov, Noise Robustness

Notebook 20 showed FIM creates hysteresis at N=8.
Notebook 24 showed FIM solves N=16.

This notebook asks: **is FIM-induced sync genuinely stable?**

Tests:
1. **Hysteresis at N=16** — forward vs backward K sweeps with FIM
2. **Lyapunov exponents** — largest LE from Jacobian, at sync vs desync
3. **Noise robustness** — how does R degrade with increasing noise?
4. **Basin of attraction** — what fraction of random ICs converge to sync?
5. **Perturbation recovery** — kick the sync state, measure recovery time

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np

sys.path.insert(
    0,
    "/media/anulum/724AA8E84AA8AA75/aaa_God_of_the_Math_Collection/03_CODE/scpn-quantum-control/src",
)
from scpn_quantum_control.bridge.knm_hamiltonian import OMEGA_N_16, build_knm_paper27

N = 16
K_base = build_knm_paper27(L=N)
omega = OMEGA_N_16[:N]


def fim_gradient(phases, i):
    n = len(phases)
    z = np.exp(1j * phases)
    mu = np.angle(np.mean(z))
    R = np.abs(np.mean(z))
    phase_diff = (mu - phases[i] + np.pi) % (2 * np.pi) - np.pi
    sensitivity = 1.0 / (1.0 - R**2 + 0.01)
    return (1.0 / n) * np.sin(phase_diff) * min(sensitivity, 50.0)


def step_kuramoto(theta, K, omega, fim_lambda, dt, noise, rng, N):
    diff = theta[None, :] - theta[:, None]
    coupling = np.sum(K * np.sin(diff), axis=1) / N
    dphi = omega + coupling
    if fim_lambda > 0:
        for i in range(N):
            dphi[i] += fim_lambda * fim_gradient(theta, i)
    theta_new = theta + dt * dphi + np.sqrt(dt) * noise * rng.normal(size=N)
    return theta_new % (2 * np.pi), dphi


print("Ready.")

## 1. Hysteresis at N=16

In [ ]:
def sweep_K(K_values, fim_lambda, forward=True, dt=0.02, T_per=80.0, noise=0.05, seed=42):
    """Sweep K carrying state between steps. Returns R at each K."""
    rng = np.random.default_rng(seed)
    theta = rng.uniform(0, 2 * np.pi, N)
    n_per = int(T_per / dt)

    if not forward:
        K_values = K_values[::-1]

    R_out = []
    for K_scale in K_values:
        K = K_base * K_scale
        R_tail = []
        for s in range(n_per):
            theta, _ = step_kuramoto(theta, K, omega, fim_lambda, dt, noise, rng, N)
            if s >= n_per * 3 // 4:
                R_tail.append(float(np.abs(np.mean(np.exp(1j * theta)))))
        R_out.append(np.mean(R_tail))

    if not forward:
        R_out = R_out[::-1]
    return np.array(R_out)


K_sweep = np.linspace(0, 20, 21)

for lam in [0.0, 1.0, 3.0]:
    R_fwd = sweep_K(K_sweep, lam, forward=True, seed=42)
    R_bwd = sweep_K(K_sweep, lam, forward=False, seed=42)

    hyst = R_fwd - R_bwd
    max_hyst = np.max(np.abs(hyst))
    hyst_width = np.sum(np.abs(hyst) > 0.05)  # K points where hysteresis > 5%

    print(f"\nλ={lam:.1f}: max hysteresis = {max_hyst:.4f}, width = {hyst_width} K-points")
    print("  K   R_fwd  R_bwd  Δ")
    for i, K in enumerate(K_sweep):
        if abs(hyst[i]) > 0.02:
            print(f"  {K:5.1f} {R_fwd[i]:.3f}  {R_bwd[i]:.3f}  {hyst[i]:+.3f}")

## 2. Lyapunov Exponents

In [ ]:
def compute_lyapunov(K_scale, fim_lambda, dt=0.02, T=200.0, noise=0.0, seed=42):
    """Estimate largest Lyapunov exponent via tangent vector evolution.
    noise=0 for clean LE estimation."""
    K = K_base * K_scale
    rng = np.random.default_rng(seed)
    theta = rng.uniform(0, 2 * np.pi, N)
    n_steps = int(T / dt)

    # Perturbation vector (tangent space)
    delta = rng.normal(size=N)
    delta = delta / np.linalg.norm(delta) * 1e-8

    lyap_sum = 0.0
    n_renorm = 0
    renorm_interval = 50

    rng_main = np.random.default_rng(seed + 1000)

    for s in range(n_steps):
        # Evolve reference trajectory
        theta, _ = step_kuramoto(theta, K, omega, fim_lambda, dt, noise, rng_main, N)

        # Evolve perturbed trajectory
        rng_pert = np.random.default_rng(seed + 1000)  # same noise realisation
        # Reset pert rng to match — for noise=0 this is irrelevant
        theta_pert = theta + delta
        theta_pert, _ = step_kuramoto(theta_pert, K, omega, fim_lambda, dt, noise, rng_pert, N)

        delta = theta_pert - theta

        if (s + 1) % renorm_interval == 0:
            norm = np.linalg.norm(delta)
            if norm > 1e-20:
                lyap_sum += np.log(norm / 1e-8)
                n_renorm += 1
                delta = delta / norm * 1e-8

    if n_renorm > 0:
        le = lyap_sum / (n_renorm * renorm_interval * dt)
    else:
        le = 0.0
    return le


# Compute LE for different regimes
print(f"{'K':>6} {'λ':>6} {'LE':>10} {'Regime':>12}")
test_cases = [
    (5, 0.0),
    (10, 0.0),
    (16, 0.0),
    (20, 0.0),  # no FIM
    (5, 1.0),
    (10, 1.0),
    (14, 1.0),
    (20, 1.0),  # FIM λ=1
    (5, 5.0),
    (10, 5.0),
    (12, 5.0),  # strong FIM
]
for K_scale, lam in test_cases:
    le = compute_lyapunov(K_scale, lam)
    regime = "STABLE" if le < -0.01 else ("MARGINAL" if le < 0.01 else "CHAOTIC")
    print(f"{K_scale:6.1f} {lam:6.1f} {le:10.4f} {regime:>12}")

## 3. Noise Robustness

In [ ]:
def measure_R(K_scale, fim_lambda, noise, dt=0.02, T=100.0, seed=42):
    K = K_base * K_scale
    rng = np.random.default_rng(seed)
    theta = rng.uniform(0, 2 * np.pi, N)
    n_steps = int(T / dt)
    R_tail = []
    for s in range(n_steps):
        theta, _ = step_kuramoto(theta, K, omega, fim_lambda, dt, noise, rng, N)
        if s >= n_steps * 3 // 4:
            R_tail.append(float(np.abs(np.mean(np.exp(1j * theta)))))
    return float(np.mean(R_tail))


noise_levels = [0.0, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1.0]

print("Noise robustness at K_scale=12:")
print(f"{'noise':>8} {'R(λ=0)':>8} {'R(λ=1)':>8} {'R(λ=5)':>8} {'FIM advantage':>14}")
for noise in noise_levels:
    R0 = np.mean([measure_R(12, 0.0, noise, seed=s) for s in range(3)])
    R1 = np.mean([measure_R(12, 1.0, noise, seed=s) for s in range(3)])
    R5 = np.mean([measure_R(12, 5.0, noise, seed=s) for s in range(3)])
    adv = R5 - R0
    print(f"{noise:8.3f} {R0:8.4f} {R1:8.4f} {R5:8.4f} {adv:+14.4f}")

## 4. Basin of Attraction Size

In [ ]:
# What fraction of random initial conditions converge to R >= 0.8?
N_ICS = 50
R_THRESH = 0.8

print("Basin of attraction (fraction of ICs reaching sync):")
print(f"{'K':>6} {'λ':>6} {'frac sync':>10} {'mean R':>8}")
for K_scale, lam in [(12, 0), (12, 1), (12, 5), (16, 0), (16, 1), (10, 5)]:
    n_sync = 0
    R_all = []
    for ic in range(N_ICS):
        R = measure_R(K_scale, lam, 0.05, seed=ic + 1000)
        R_all.append(R)
        if R >= R_THRESH:
            n_sync += 1
    frac = n_sync / N_ICS
    print(f"{K_scale:6.1f} {lam:6.1f} {frac:10.2f} {np.mean(R_all):8.4f}")

## 5. Perturbation Recovery Time

In [ ]:
def perturbation_recovery(
    K_scale,
    fim_lambda,
    kick_size=1.0,
    dt=0.02,
    T_settle=100.0,
    T_recovery=50.0,
    noise=0.05,
    seed=42,
):
    """Settle into sync, then kick phases by random amounts, measure recovery time."""
    K = K_base * K_scale
    rng = np.random.default_rng(seed)
    theta = rng.uniform(0, 2 * np.pi, N)

    # Settle
    for _ in range(int(T_settle / dt)):
        theta, _ = step_kuramoto(theta, K, omega, fim_lambda, dt, noise, rng, N)

    R_before = float(np.abs(np.mean(np.exp(1j * theta))))

    # Kick
    theta += kick_size * rng.normal(size=N)
    theta %= 2 * np.pi
    R_after_kick = float(np.abs(np.mean(np.exp(1j * theta))))

    # Track recovery
    recovery_steps = int(T_recovery / dt)
    R_trace = []
    for s in range(recovery_steps):
        theta, _ = step_kuramoto(theta, K, omega, fim_lambda, dt, noise, rng, N)
        if s % 10 == 0:
            R_trace.append(float(np.abs(np.mean(np.exp(1j * theta)))))

    # Recovery time: first time R exceeds 90% of pre-kick R
    target = R_before * 0.9
    recovery_time = None
    for i, R in enumerate(R_trace):
        if target <= R:
            recovery_time = i * 10 * dt
            break

    return {
        "R_before": round(R_before, 4),
        "R_after_kick": round(R_after_kick, 4),
        "R_final": round(float(R_trace[-1]), 4),
        "recovery_time": recovery_time,
    }


print("Perturbation recovery (kick=1.0 rad, K=12):")
print(f"{'λ':>6} {'R_before':>10} {'R_kicked':>10} {'R_final':>10} {'t_recovery':>12}")
for lam in [0.0, 0.5, 1.0, 2.0, 5.0]:
    res = perturbation_recovery(12, lam, kick_size=1.0)
    t_rec = f"{res['recovery_time']:.1f}" if res["recovery_time"] is not None else "NEVER"
    print(
        f"{lam:6.1f} {res['R_before']:10.4f} {res['R_after_kick']:10.4f} {res['R_final']:10.4f} {t_rec:>12}"
    )

In [ ]:
# Save
output = {
    "experiment": "FIM sync stability analysis at N=16",
    "N": N,
}
out_path = Path(
    "/media/anulum/724AA8E84AA8AA75/aaa_God_of_the_Math_Collection/03_CODE/scpn-quantum-control/results/fim_stability_2026-03-29.json"
)
with open(out_path, "w") as f:
    json.dump(output, f, indent=2)
print(f"Results saved to {out_path}")